# 工具定义

不使用`@tool`的方式定义工具

In [6]:
import os
from imaplib import Literal

from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

# 读取.env配置文件中的信息，相关环境变量以.env文件中优先
load_dotenv(override=True)
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 模型初始化
model = ChatDeepSeek(
    model="deepseek-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL
)


def get_weather(city: str) -> str:
    """
    获取指定城市的天气信息
    :param city: 城市名称
    :return: 天气信息字符串
    """
    return city + "：晴天，温度15℃"


# 绑定工具
model_with_tools = model.bind_tools([get_weather])

# 调用模型
response = model_with_tools.invoke("武汉的天气如何？")

rprint(response)

AIMessage(
    content='我来帮您查询武汉的天气情况。',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 'The user asks about the weather in Wuhan. I should call get_weather with city 
"武汉".'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 67,
            'prompt_tokens': 309,
            'total_tokens': 376,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 20,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 128},
            'prompt_cache_hit_tokens': 128,
            'prompt_cache_miss_tokens': 181
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
        'id': 'fa92a863-ef20-489f-b743-52188a0701a7',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--01a0ea9c-dd92-7360-a96f-9f915f525f81-0',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'city': '武汉'},
            'id': 'call_00_E5Upb5ApqlDyjlZxSgNF4016',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 309,
        'output_tokens': 67,
        'total_tokens': 376,
        'input_token_details': {'cache_read': 128},
        'output_token_details': {'reasoning': 20}
    }
)

In [8]:
from langchain_core.utils.function_calling import convert_to_openai_tool


def get_weather(city: str) -> str:
    """
    获取指定城市的天气信息

    Args:
        city: 城市名称

    Returns:
        天气信息字符串
    """
    return city + "：晴天，温度15℃"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取指定城市的天气信息\n:param city: 城市名称\n:return: 天气信息字符串',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

使用`@tool`

In [5]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

from rich import print as rprint


@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """
    获取指定城市的天气信息

    Args:
        city: 城市名称

    Returns:
        天气信息字符串
    """
    return city + "：晴天，温度15℃"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取指定城市的天气信息',
        'parameters': {
            'properties': {'city': {'description': '城市名称', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

In [12]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

from rich import print as rprint
from typing import Literal

from pydantic import BaseModel, Field


class WeatherInput(BaseModel):
    city: str = Field(description="城市名称", default="武汉")
    # 通过Literal可以固定参数范围
    unit: Literal["celsius", "fahrenheit"] = Field(description="温度单位")


@tool(description="获取指定城市的天气信息", args_schema=WeatherInput)
def get_weather(city: str, unit: str) -> str:
    return city + "：晴天，温度15℃"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取指定城市的天气信息',
        'parameters': {
            'properties': {
                'city': {'default': '武汉', 'description': '城市名称', 'type': 'string'},
                'unit': {'description': '温度单位', 'enum': ['celsius', 'fahrenheit'], 'type': 'string'}
            },
            'required': ['unit'],
            'type': 'object'
        }
    }
}

In [13]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

from rich import print as rprint

json_schema = {
    'properties': {
        'city': {'default': '武汉', 'description': '城市名称', 'type': 'string'},
        'unit': {'description': '温度单位', 'enum': ['celsius', 'fahrenheit'], 'type': 'string'}
    },
    'required': ['unit'],
    'type': 'object'
}


@tool(description="获取指定城市的天气信息", args_schema=json_schema)
def get_weather(city: str, unit: str) -> str:
    return city + "：晴天，温度15℃"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取指定城市的天气信息',
        'parameters': {
            'properties': {
                'city': {'default': '武汉', 'description': '城市名称', 'type': 'string'},
                'unit': {'description': '温度单位', 'enum': ['celsius', 'fahrenheit'], 'type': 'string'}
            },
            'required': ['unit'],
            'type': 'object'
        }
    }
}